# 📊 Relatório de Análise de Recursos Operacionais

**OptiFlow — G4 AI Master Challenge 002: Redesign de Suporte**

---

| Item | Detalhe |
|------|---------|
| **Dataset** | Customer Support Tickets — 8.469 tickets totais |
| **Amostra analisada** | 2.769 tickets fechados com CSAT |
| **Período** | Dados completos do dataset |
| **Métrica-chave** | `duration_hours = abs(TTR - FRT)` |
| **Framework** | Diagnóstico de 6 Cenários (Canal × Assunto) |
| **Data do relatório** | 21 de março de 2026 |

---

> **Objetivo:** Mapear para ONDE vão as horas operacionais, classificar cada par Canal × Assunto em 6 cenários de ação, e quantificar o desperdício para propor realocação de recursos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import pearsonr
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

SCENARIO_COLORS = {
    'acelerar': '#ef4444',
    'desacelerar': '#3b82f6',
    'redirecionar': '#f97316',
    'quarentena': '#eab308',
    'manter': '#22c55e',
    'liberar': '#9ca3af',
}

SCENARIO_LABELS = {
    'acelerar': '🔴 Acelerar',
    'desacelerar': '🔵 Desacelerar',
    'redirecionar': '🟠 Redirecionar',
    'quarentena': '🟡 Quarentena',
    'manter': '🟢 Manter',
    'liberar': '⚪ Liberar',
}

CHANNEL_COLORS = {'Chat': '#3b82f6', 'Email': '#10b981', 'Phone': '#f97316', 'Social media': '#8b5cf6'}

TABLE_STYLES = [
    {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('margin-bottom', '8px')]},
    {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'), ('padding', '8px 12px'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('text-align', 'center')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f8fafc')]},
    {'selector': 'tr:hover', 'props': [('background-color', '#e2e8f0')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('margin', '10px 0')]},
]

def styled_table(df, caption=''):
    """Render a styled HTML table."""
    return df.style.set_caption(caption).set_table_styles(TABLE_STYLES)

print("✅ Bibliotecas carregadas e estilos configurados.")

✅ Bibliotecas carregadas e estilos configurados.


In [2]:
# === CARGA DE DADOS ===
csv_path = 'data/customer_support_tickets.csv'
df_raw = pd.read_csv(csv_path)

df = df_raw[df_raw['Ticket Status'] == 'Closed'].copy()
df['First Response Time'] = pd.to_datetime(df['First Response Time'], errors='coerce')
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'], errors='coerce')
df['duration_hours'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds().abs() / 3600
df = df.dropna(subset=['Customer Satisfaction Rating', 'duration_hours'])
df = df.rename(columns={
    'Ticket Channel': 'channel',
    'Ticket Subject': 'subject',
    'Customer Satisfaction Rating': 'csat',
})

TOTAL_HOURS = df['duration_hours'].sum()

# Summary card
display(HTML(f"""
<div style="display:flex; gap:16px; margin:16px 0;">
  <div style="flex:1; background:#1e293b; color:white; padding:20px; border-radius:12px; text-align:center;">
    <div style="font-size:13px; opacity:0.7;">Total de Tickets</div>
    <div style="font-size:32px; font-weight:bold;">{len(df_raw):,}</div>
    <div style="font-size:12px; opacity:0.5;">dataset completo</div>
  </div>
  <div style="flex:1; background:#1e293b; color:white; padding:20px; border-radius:12px; text-align:center;">
    <div style="font-size:13px; opacity:0.7;">Fechados com CSAT</div>
    <div style="font-size:32px; font-weight:bold;">{len(df):,}</div>
    <div style="font-size:12px; opacity:0.5;">amostra analisada</div>
  </div>
  <div style="flex:1; background:#1e293b; color:white; padding:20px; border-radius:12px; text-align:center;">
    <div style="font-size:13px; opacity:0.7;">Horas Operacionais</div>
    <div style="font-size:32px; font-weight:bold;">{TOTAL_HOURS:,.0f}h</div>
    <div style="font-size:12px; opacity:0.5;">abs(TTR - FRT)</div>
  </div>
  <div style="flex:1; background:#1e293b; color:white; padding:20px; border-radius:12px; text-align:center;">
    <div style="font-size:13px; opacity:0.7;">CSAT Médio</div>
    <div style="font-size:32px; font-weight:bold;">{df['csat'].mean():.2f}</div>
    <div style="font-size:12px; opacity:0.5;">escala 1-5</div>
  </div>
  <div style="flex:1; background:#1e293b; color:white; padding:20px; border-radius:12px; text-align:center;">
    <div style="font-size:13px; opacity:0.7;">Pares Analisados</div>
    <div style="font-size:32px; font-weight:bold;">64</div>
    <div style="font-size:12px; opacity:0.5;">Canal × Assunto</div>
  </div>
</div>
"""))

FileNotFoundError: [Errno 2] No such file or directory: 'data/customer_support_tickets.csv'

---

## 1. Metodologia

### Proxy de Esforço Operacional

Usamos `duration_hours = abs(TTR - FRT)` como proxy de quanto tempo operacional cada ticket consumiu — do momento da primeira resposta até a resolução final.

> **Por que não usar TTR diretamente?** O TTR inclui tempo de espera do cliente. A diferença `TTR - FRT` isola melhor o tempo de trabalho do agente. O `abs()` corrige inconsistências nos timestamps do dataset sintético.

### Framework de 6 Cenários

Cada par Canal × Assunto (mínimo 5 tickets) é classificado com base em:

| # | Cenário | Critério | Ação Recomendada |
|:-:|---------|----------|------------------|
| 1 | 🔴 **Acelerar** | Pearson r < -0.3 (mais tempo → pior CSAT) | Investir em SLA agressivo, automação para reduzir tempo |
| 2 | 🔵 **Desacelerar** | Pearson r > 0.3 (mais tempo → melhor CSAT) | Manter ritmo, agentes seniores, não apressar |
| 3 | 🟠 **Redirecionar** | Gap CSAT > 0.5 + canal alternativo viável | Mover tickets para canal com melhor CSAT |
| 4 | 🟡 **Quarentena** | Gap CSAT > 0.5, sem alvo viável / todos ruins | Investigar causa raiz antes de agir |
| 5 | 🟢 **Manter** | CSAT ≥ 3.5, sem correlação forte | Preservar — já funciona bem |
| 6 | ⚪ **Liberar** | Nenhum dos anteriores | Recursos sem retorno claro — candidatos a realocação |

### Matriz de Viabilidade de Redirecionamento

| De ↓ / Para → | Email | Phone | Chat | Social Media |
|:-:|:-:|:-:|:-:|:-:|
| **Email** | — | ✅ | ✅ | ❌ |
| **Phone** | ✅ | — | ✅ | ❌ |
| **Chat** | ✅ | ✅ | — | ❌ |
| **Social Media** | ✅ | ✅ | ❌ | — |

---

## 2. Distribuição de Horas por Canal

In [ ]:
# --- HORAS POR CANAL ---
ch_stats = df.groupby('channel').agg(
    total_hours=('duration_hours', 'sum'),
    avg_hours=('duration_hours', 'mean'),
    median_hours=('duration_hours', 'median'),
    n_tickets=('duration_hours', 'count'),
    avg_csat=('csat', 'mean'),
).sort_values('total_hours', ascending=False)

ch_stats['pct'] = ch_stats['total_hours'] / TOTAL_HOURS * 100

# Table
ch_display = ch_stats.copy()
ch_display.index.name = 'Canal'
ch_display.columns = ['Horas Totais', 'Média (h)', 'Mediana (h)', 'Tickets', 'CSAT Médio', '% do Total']
ch_display['Horas Totais'] = ch_display['Horas Totais'].apply(lambda x: f'{x:,.0f}h')
ch_display['Média (h)'] = ch_display['Média (h)'].apply(lambda x: f'{x:.1f}')
ch_display['Mediana (h)'] = ch_display['Mediana (h)'].apply(lambda x: f'{x:.1f}')
ch_display['Tickets'] = ch_display['Tickets'].astype(int)
ch_display['CSAT Médio'] = ch_display['CSAT Médio'].apply(lambda x: f'{x:.2f}')
ch_display['% do Total'] = ch_display['% do Total'].apply(lambda x: f'{x:.1f}%')

display(styled_table(ch_display, caption='Tabela 1 — Horas Operacionais por Canal'))

# Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
bars = ax1.barh(ch_stats.index[::-1], ch_stats['total_hours'][::-1],
                color=[CHANNEL_COLORS.get(c, '#888') for c in ch_stats.index[::-1]],
                edgecolor='white', linewidth=1.5)
for bar, (ch, row) in zip(bars, ch_stats[::-1].iterrows()):
    ax1.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
             f'{row["total_hours"]:,.0f}h ({row["pct"]:.1f}%)', va='center', fontsize=11, fontweight='bold')
ax1.set_xlabel('Horas Operacionais')
ax1.set_title('Horas por Canal')
ax1.set_xlim(0, ch_stats['total_hours'].max() * 1.35)
ax1.grid(axis='y', alpha=0)

# Pie chart
ax2.pie(ch_stats['total_hours'], labels=ch_stats.index,
        colors=[CHANNEL_COLORS.get(c, '#888') for c in ch_stats.index],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
ax2.set_title('Distribuição Proporcional')

plt.tight_layout()
plt.savefig('process-log/screenshots/p9_hours_by_channel.png', dpi=150, bbox_inches='tight')
plt.show()

display(Markdown("""
> **Observação:** Distribuição quase uniforme entre os 4 canais (~25% cada).
> Em operações reais, esperaríamos concentração em 1-2 canais dominantes.
> A uniformidade é característica de dados sintéticos.
"""))

---

## 3. Distribuição de Horas por Assunto

In [ ]:
# --- HORAS POR ASSUNTO ---
sub_stats = df.groupby('subject').agg(
    total_hours=('duration_hours', 'sum'),
    avg_hours=('duration_hours', 'mean'),
    n_tickets=('duration_hours', 'count'),
    avg_csat=('csat', 'mean'),
).sort_values('total_hours', ascending=False)

sub_stats['pct'] = sub_stats['total_hours'] / TOTAL_HOURS * 100

sub_display = sub_stats.copy()
sub_display.index.name = 'Assunto'
sub_display.columns = ['Horas Totais', 'Média (h)', 'Tickets', 'CSAT Médio', '% do Total']
sub_display['Horas Totais'] = sub_display['Horas Totais'].apply(lambda x: f'{x:,.0f}h')
sub_display['Média (h)'] = sub_display['Média (h)'].apply(lambda x: f'{x:.1f}')
sub_display['Tickets'] = sub_display['Tickets'].astype(int)
sub_display['CSAT Médio'] = sub_display['CSAT Médio'].apply(lambda x: f'{x:.2f}')
sub_display['% do Total'] = sub_display['% do Total'].apply(lambda x: f'{x:.1f}%')

display(styled_table(sub_display, caption='Tabela 2 — Horas Operacionais por Assunto (16 categorias)'))

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sub_stats)))
bars = ax.barh(sub_stats.index[::-1], sub_stats['total_hours'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.8)
for bar, (sub, row) in zip(bars, sub_stats[::-1].iterrows()):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{row["total_hours"]:,.0f}h ({row["pct"]:.1f}%) — CSAT {row["avg_csat"]:.2f}',
            va='center', fontsize=9)
ax.set_xlabel('Horas Operacionais')
ax.set_title('Horas Operacionais por Assunto')
ax.set_xlim(0, sub_stats['total_hours'].max() * 1.45)
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig('process-log/screenshots/p9_hours_by_subject.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 4. Heatmaps: Horas e CSAT por Par Canal × Assunto

Dois heatmaps lado a lado: à esquerda, horas operacionais (vermelho = mais horas); à direita, CSAT médio (verde = mais satisfeito, vermelho = menos).

In [ ]:
# --- HEATMAPS CANAL x ASSUNTO ---
pair_stats = df.groupby(['channel', 'subject']).agg(
    total_hours=('duration_hours', 'sum'),
    avg_csat=('csat', 'mean'),
    n_tickets=('duration_hours', 'count'),
).reset_index()

pivot_hours = pair_stats.pivot(index='subject', columns='channel', values='total_hours').fillna(0)
pivot_csat = pair_stats.pivot(index='subject', columns='channel', values='avg_csat')

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

sns.heatmap(pivot_hours, annot=True, fmt=',.0f', cmap='YlOrRd', ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Horas'})
axes[0].set_title('Horas Operacionais por Par')
axes[0].set_xlabel('Canal')
axes[0].set_ylabel('Assunto')

sns.heatmap(pivot_csat, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1],
            linewidths=0.5, vmin=1, vmax=5, cbar_kws={'label': 'CSAT Médio'})
axes[1].set_title('CSAT Médio por Par')
axes[1].set_xlabel('Canal')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('process-log/screenshots/p9_heatmaps_hours_csat.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 10 pairs table
top10 = pair_stats.sort_values('total_hours', ascending=False).head(10).copy()
top10['pct'] = top10['total_hours'] / TOTAL_HOURS * 100
top10_display = top10[['channel', 'subject', 'total_hours', 'n_tickets', 'avg_csat', 'pct']].copy()
top10_display.columns = ['Canal', 'Assunto', 'Horas', 'Tickets', 'CSAT', '% Total']
top10_display['Horas'] = top10_display['Horas'].apply(lambda x: f'{x:,.0f}h')
top10_display['CSAT'] = top10_display['CSAT'].apply(lambda x: f'{x:.2f}')
top10_display['% Total'] = top10_display['% Total'].apply(lambda x: f'{x:.1f}%')
top10_display = top10_display.reset_index(drop=True)
top10_display.index = range(1, 11)
top10_display.index.name = '#'

display(styled_table(top10_display, caption='Tabela 3 — Top 10 Pares por Consumo de Horas'))

---

## 5. Classificação por Cenário — A Descoberta Central

Aplicamos o framework de 6 cenários a cada um dos 64 pares Canal × Assunto. O resultado revela onde a operação aloca recursos de forma eficiente — e onde desperdiça.

In [ ]:
# === CLASSIFICACAO DOS 64 PARES ===
REDIRECT_VIABLE = {
    'Email': {'Phone': True, 'Chat': True, 'Social media': False},
    'Chat': {'Phone': True, 'Email': True, 'Social media': False},
    'Phone': {'Email': True, 'Chat': True, 'Social media': False},
    'Social media': {'Email': True, 'Phone': True, 'Chat': False},
}

channels = sorted(df['channel'].unique())
subjects = sorted(df['subject'].unique())

# Pre-compute per-subject channel CSAT
subj_ch_csat = {}
for sub in subjects:
    ch_stats_list = []
    for ch in channels:
        subset = df[(df['channel'] == ch) & (df['subject'] == sub)]
        if len(subset) >= 5:
            ch_stats_list.append({'channel': ch, 'avg_csat': round(subset['csat'].mean(), 2)})
    subj_ch_csat[sub] = sorted(ch_stats_list, key=lambda x: -x['avg_csat'])

diags = []
for ch in channels:
    for sub in subjects:
        subset = df[(df['channel'] == ch) & (df['subject'] == sub)]
        if len(subset) < 5:
            continue
        r_val, _ = pearsonr(subset['duration_hours'], subset['csat'])
        r_val = round(r_val, 3)
        pair_csat = round(subset['csat'].mean(), 2)
        pair_hours = subset['duration_hours'].sum()
        n = len(subset)
        
        chs = subj_ch_csat.get(sub, [])
        best_ch = chs[0] if chs else None
        redirect_to = None
        
        if r_val < -0.3:
            scenario = 'acelerar'
        elif r_val > 0.3:
            scenario = 'desacelerar'
        else:
            gap = best_ch['avg_csat'] - pair_csat if best_ch else 0
            if gap > 0.5 and best_ch and best_ch['channel'] != ch:
                viable = [c for c in chs
                          if c['channel'] != ch
                          and c['avg_csat'] > pair_csat + 0.3
                          and REDIRECT_VIABLE.get(ch, {}).get(c['channel'], False)]
                if viable:
                    scenario = 'redirecionar'
                    redirect_to = viable[0]['channel']
                else:
                    scenario = 'quarentena'
            elif pair_csat >= 3.5:
                scenario = 'manter'
            else:
                all_bad = all(c['avg_csat'] < 3.0 for c in chs)
                scenario = 'quarentena' if (all_bad and pair_csat < 3.0) else 'liberar'
        
        diags.append({
            'channel': ch, 'subject': sub, 'scenario': scenario,
            'r_pair': r_val, 'pair_csat': pair_csat,
            'total_hours': round(pair_hours, 1), 'n_tickets': n,
            'redirect_to': redirect_to,
        })

diag_df = pd.DataFrame(diags)

# === TABELA RESUMO POR CENARIO ===
scenario_order = ['acelerar', 'desacelerar', 'redirecionar', 'quarentena', 'manter', 'liberar']
rows = []
for sc in scenario_order:
    sd = diag_df[diag_df['scenario'] == sc]
    if len(sd) == 0:
        continue
    h = sd['total_hours'].sum()
    w_csat = np.average(sd['pair_csat'], weights=sd['total_hours'])
    rows.append({
        'Cenário': SCENARIO_LABELS[sc],
        'Pares': len(sd),
        'Tickets': sd['n_tickets'].sum(),
        'Horas': f'{h:,.0f}h',
        '% do Total': f'{h/TOTAL_HOURS*100:.1f}%',
        'CSAT Médio': f'{w_csat:.2f}',
    })
rows.append({
    'Cenário': '**TOTAL**',
    'Pares': len(diag_df),
    'Tickets': diag_df['n_tickets'].sum(),
    'Horas': f'{TOTAL_HOURS:,.0f}h',
    '% do Total': '100.0%',
    'CSAT Médio': '—',
})

summary_df = pd.DataFrame(rows)
display(styled_table(summary_df.set_index('Cenário'), caption='Tabela 4 — Distribuição de Recursos por Cenário'))

# Hero metric
liberar_hours = diag_df[diag_df['scenario'] == 'liberar']['total_hours'].sum()
liberar_pct = liberar_hours / TOTAL_HOURS * 100

display(HTML(f"""
<div style="margin:24px 0; padding:32px; background:linear-gradient(135deg, #dc2626 0%, #991b1b 100%);
            color:white; border-radius:16px; text-align:center;">
  <div style="font-size:16px; opacity:0.8; margin-bottom:8px;">MÉTRICA-CHAVE: DESPERDÍCIO IDENTIFICADO</div>
  <div style="font-size:56px; font-weight:bold; line-height:1.1;">{liberar_pct:.1f}%</div>
  <div style="font-size:20px; margin-top:8px;">{liberar_hours:,.0f} horas em pares "Liberar"</div>
  <div style="font-size:14px; opacity:0.7; margin-top:12px;">
    Recursos sem retorno claro em satisfação — candidatos imediatos a realocação ou automação
  </div>
</div>
"""))

In [ ]:
# === SCATTER: MAPA DE CENARIOS ===
fig, ax = plt.subplots(figsize=(14, 8))

for sc in scenario_order:
    sd = diag_df[diag_df['scenario'] == sc]
    if len(sd) == 0:
        continue
    ax.scatter(sd['r_pair'], sd['pair_csat'],
               c=SCENARIO_COLORS[sc], label=f'{sc} ({len(sd)} pares)',
               s=sd['total_hours'] / diag_df['total_hours'].max() * 500 + 40,
               alpha=0.75, edgecolors='black', linewidths=0.5, zorder=3)

# Reference lines
ax.axvline(x=-0.3, color='#ef4444', linestyle=':', alpha=0.6, linewidth=1.5)
ax.axvline(x=0.3, color='#3b82f6', linestyle=':', alpha=0.6, linewidth=1.5)
ax.axhline(y=3.5, color='#22c55e', linestyle=':', alpha=0.6, linewidth=1.5)

# Zone labels
ax.text(-0.65, 4.8, 'ACELERAR\n(r < -0.3)', fontsize=9, color='#ef4444', fontweight='bold', ha='center')
ax.text(0.65, 4.8, 'DESACELERAR\n(r > 0.3)', fontsize=9, color='#3b82f6', fontweight='bold', ha='center')
ax.text(0, 3.7, 'MANTER (CSAT >= 3.5)', fontsize=9, color='#22c55e', fontweight='bold', ha='center')

ax.set_xlabel('Pearson r (duração × CSAT)', fontsize=12)
ax.set_ylabel('CSAT Médio do Par', fontsize=12)
ax.set_title('Mapa de Cenários: Correlação vs Satisfação (tamanho = horas operacionais)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, fontsize=10)
ax.set_xlim(-0.85, 0.95)
ax.set_ylim(1.5, 5.0)
plt.tight_layout()
plt.savefig('process-log/screenshots/p9_scenario_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. Detalhamento dos Pares Acionáveis

Os 14 pares que **não** são "Liberar" — estes são os que têm ação concreta recomendada.

In [ ]:
# === PARES ACIONAVEIS (nao-liberar) ===
SCENARIO_ACTIONS = {
    'acelerar': 'Reduzir tempo de resolução — SLA agressivo, automação',
    'desacelerar': 'Manter ritmo — agentes seniores, não apressar',
    'redirecionar': 'Mover para canal com melhor CSAT',
    'quarentena': 'Investigar causa raiz — todos os canais ruins',
    'manter': 'Preservar como está — já funciona bem',
}

actionable = diag_df[diag_df['scenario'] != 'liberar'].sort_values(
    ['scenario', 'total_hours'], ascending=[True, False]
).copy()

act_display = actionable[['channel', 'subject', 'scenario', 'r_pair', 'pair_csat', 'total_hours', 'n_tickets', 'redirect_to']].copy()
act_display.columns = ['Canal', 'Assunto', 'Cenário', 'Pearson r', 'CSAT', 'Horas', 'Tickets', 'Redirecionar Para']
act_display['Cenário'] = act_display['Cenário'].map(SCENARIO_LABELS)
act_display['Horas'] = act_display['Horas'].apply(lambda x: f'{x:,.0f}h')
act_display['CSAT'] = act_display['CSAT'].apply(lambda x: f'{x:.2f}')
act_display['Pearson r'] = act_display['Pearson r'].apply(lambda x: f'{x:.3f}')
act_display['Redirecionar Para'] = act_display['Redirecionar Para'].fillna('—')
act_display = act_display.reset_index(drop=True)
act_display.index = range(1, len(act_display) + 1)
act_display.index.name = '#'

display(styled_table(act_display, caption='Tabela 5 — Pares Acionáveis: Detalhamento Completo'))

# Summary per actionable scenario
display(Markdown("### Resumo por cenário acionável"))
for sc in ['acelerar', 'desacelerar', 'redirecionar', 'quarentena', 'manter']:
    sd = diag_df[diag_df['scenario'] == sc]
    if len(sd) == 0:
        continue
    h = sd['total_hours'].sum()
    display(HTML(f"""
    <div style="margin:8px 0; padding:12px 16px; border-left:4px solid {SCENARIO_COLORS[sc]};
                background:{SCENARIO_COLORS[sc]}11; border-radius:0 8px 8px 0;">
      <strong>{SCENARIO_LABELS[sc]}</strong> — {len(sd)} pares, {h:,.0f}h ({h/TOTAL_HOURS*100:.1f}%)<br/>
      <span style="color:#555;">{SCENARIO_ACTIONS[sc]}</span>
    </div>
    """))

---

## 7. Cenários de Realocação de Recursos

Se movermos uma fração das horas "Liberar" (desperdício) para reforçar os pares "Acelerar" (onde tempo é crítico), qual o impacto potencial?

In [ ]:
# === CENARIOS DE REALOCACAO ===
acelerar_hours = diag_df[diag_df['scenario'] == 'acelerar']['total_hours'].sum()
outros_hours = TOTAL_HOURS - acelerar_hours - liberar_hours

scenarios_pct = [0.25, 0.50, 0.75]
realloc_rows = []
for pct in scenarios_pct:
    moved = liberar_hours * pct
    new_acel = acelerar_hours + moved
    remaining = liberar_hours * (1 - pct)
    boost = moved / acelerar_hours * 100 if acelerar_hours > 0 else 0
    realloc_rows.append({
        'Cenário': f'{int(pct*100)}% Realocado',
        'Horas Movidas': f'{moved:,.0f}h',
        'Acelerar (antes)': f'{acelerar_hours:,.0f}h',
        'Acelerar (depois)': f'{new_acel:,.0f}h',
        'Aumento Capacidade': f'+{boost:,.0f}%',
        'Liberar Restante': f'{remaining:,.0f}h',
    })

realloc_df = pd.DataFrame(realloc_rows)
display(styled_table(realloc_df.set_index('Cenário'),
                     caption='Tabela 6 — Cenários de Realocação: Liberar → Acelerar'))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bars: before/after
labels = ['Atual', '25%', '50%', '75%']
acel_vals = [acelerar_hours] + [acelerar_hours + liberar_hours * p for p in scenarios_pct]
lib_vals = [liberar_hours] + [liberar_hours * (1 - p) for p in scenarios_pct]

x = np.arange(len(labels))
width = 0.5
ax1.bar(x, acel_vals, width, label='Acelerar', color=SCENARIO_COLORS['acelerar'], alpha=0.85)
ax1.bar(x, lib_vals, width, bottom=acel_vals, label='Liberar', color=SCENARIO_COLORS['liberar'], alpha=0.85)
ax1.bar(x, [outros_hours]*4, width, bottom=[a+l for a,l in zip(acel_vals, lib_vals)],
        label='Outros cenários', color='#64748b', alpha=0.5)

for i, (a, l) in enumerate(zip(acel_vals, lib_vals)):
    ax1.text(i, a/2, f'{a:,.0f}h', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if l > 500:
        ax1.text(i, a + l/2, f'{l:,.0f}h', ha='center', va='center', fontsize=9, color='#333')

ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel('Horas')
ax1.set_title('Redistribuição de Recursos por Cenário')
ax1.legend(loc='upper right')

# Donut: current distribution
sc_hours = diag_df.groupby('scenario')['total_hours'].sum()
sc_hours = sc_hours.reindex([s for s in scenario_order if s in sc_hours.index])
colors_pie = [SCENARIO_COLORS.get(s, '#9ca3af') for s in sc_hours.index]
wedges, texts, autotexts = ax2.pie(
    sc_hours.values, labels=[f'{SCENARIO_LABELS.get(s, s)}\n{h:,.0f}h' for s, h in zip(sc_hours.index, sc_hours.values)],
    colors=colors_pie, autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.5), textprops={'fontsize': 9}
)
ax2.set_title('Distribuição Atual de Horas por Cenário')

plt.tight_layout()
plt.savefig('process-log/screenshots/p9_reallocation.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. O Paradoxo de Simpson

Por que 79,5% dos pares são "Liberar" se existem correlações fortes em pares individuais?

In [ ]:
# === PARADOXO DE SIMPSON ===
# Global correlation
from scipy.stats import pearsonr as pr
r_global, p_global = pr(df['duration_hours'], df['csat'])

# Per-pair distribution of r values
r_values = diag_df['r_pair'].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: global scatter
ax1.scatter(df['duration_hours'], df['csat'], alpha=0.15, s=15, c='#64748b')
z = np.polyfit(df['duration_hours'], df['csat'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df['duration_hours'].max(), 100)
ax1.plot(x_line, p(x_line), 'r-', linewidth=2, label=f'R² = {r_global**2:.4f}')
ax1.set_xlabel('Duração (horas)')
ax1.set_ylabel('CSAT')
ax1.set_title(f'Visão Global: r = {r_global:.3f} (sem correlação)')
ax1.legend(fontsize=11)

# Right: distribution of per-pair r values
colors_hist = [SCENARIO_COLORS['acelerar'] if r < -0.3
               else SCENARIO_COLORS['desacelerar'] if r > 0.3
               else SCENARIO_COLORS['liberar'] for r in r_values]
ax2.bar(range(len(r_values)), sorted(r_values), color=[c for _, c in sorted(zip(r_values, colors_hist))],
        edgecolor='white', linewidth=0.3)
ax2.axhline(y=-0.3, color='#ef4444', linestyle='--', linewidth=1.5, label='Limite Acelerar (r < -0.3)')
ax2.axhline(y=0.3, color='#3b82f6', linestyle='--', linewidth=1.5, label='Limite Desacelerar (r > 0.3)')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
ax2.set_xlabel('Pares Canal × Assunto (ordenados por r)')
ax2.set_ylabel('Pearson r')
ax2.set_title(f'Visão Por Par: r varia de {r_values.min():.2f} a {r_values.max():.2f}')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('process-log/screenshots/p9_simpson_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML(f"""
<div style="margin:16px 0; padding:20px; background:#fef3c7; border:1px solid #f59e0b;
            border-radius:12px;">
  <strong style="font-size:14px;">Paradoxo de Simpson — Explicação</strong><br/><br/>
  <strong>Nível global:</strong> r = {r_global:.3f}, R² = {r_global**2:.4f} → duração NÃO correlaciona com CSAT.<br/>
  <strong>Nível por par:</strong> r varia de {r_values.min():.2f} a {r_values.max():.2f} → correlações fortes existem, mas em direções opostas.<br/><br/>
  A maioria dos pares ({len(diag_df[diag_df['scenario']=='liberar'])}/64) tem correlação fraca (-0.3 < r < 0.3),
  o que explica por que são classificados como "Liberar". Apenas {len(diag_df[diag_df['scenario'].isin(['acelerar','desacelerar'])])} pares
  cruzam os limiares de correlação significativa.
</div>
"""))

---

## 9. Recomendações Operacionais

In [ ]:
# === RECOMENDACOES ===
redir_pairs = diag_df[diag_df['scenario'] == 'redirecionar']
redir_hours = redir_pairs['total_hours'].sum()
acel_pairs = diag_df[diag_df['scenario'] == 'acelerar']
quar_pairs = diag_df[diag_df['scenario'] == 'quarentena']

recommendations = [
    {
        'Prazo': 'Imediato (< 1 semana)',
        'Ação': 'Reordenar Fila de Atendimento',
        'Detalhe': f'Tickets "acelerar" ({acel_pairs["n_tickets"].sum()} tickets) vão para o topo. Tickets "liberar" podem esperar.',
        'Impacto': f'{acelerar_hours:,.0f}h priorizadas',
        'Esforço': 'Baixo',
    },
    {
        'Prazo': 'Imediato (< 1 semana)',
        'Ação': 'Auto-Roteamento por Cenário',
        'Detalhe': f'{len(redir_pairs)} pares "redirecionar" encaminhados ao canal com melhor CSAT.',
        'Impacto': f'{redir_hours:,.0f}h otimizadas',
        'Esforço': 'Baixo',
    },
    {
        'Prazo': 'Curto Prazo (1-4 sem)',
        'Ação': 'Chatbot para Pares "Liberar"',
        'Detalhe': f'Autoatendimento nos {len(diag_df[diag_df["scenario"]=="liberar"])} pares onde esforço humano não impacta CSAT.',
        'Impacto': f'Até {liberar_hours*0.5:,.0f}h liberadas',
        'Esforço': 'Médio',
    },
    {
        'Prazo': 'Curto Prazo (1-4 sem)',
        'Ação': 'Roteamento Especialista',
        'Detalhe': f'{len(quar_pairs)} pares "quarentena" ({quar_pairs["total_hours"].sum():,.0f}h) — agentes seniores investigam.',
        'Impacto': 'Diagnóstico qualitativo',
        'Esforço': 'Médio',
    },
    {
        'Prazo': 'Médio Prazo (1-3 meses)',
        'Ação': 'Dashboard de Monitoramento',
        'Detalhe': 'Classificação automática por cenário em tempo real. Alertas para mudanças de cenário.',
        'Impacto': 'Visibilidade contínua',
        'Esforço': 'Alto',
    },
    {
        'Prazo': 'Médio Prazo (1-3 meses)',
        'Ação': 'Modelo Preditivo de CSAT',
        'Detalhe': 'Antecipar insatisfação antes do fechamento. Escalar tickets em risco.',
        'Impacto': 'Prevenção proativa',
        'Esforço': 'Alto',
    },
]

rec_df = pd.DataFrame(recommendations)
rec_df.index = range(1, len(rec_df) + 1)
rec_df.index.name = '#'

display(styled_table(rec_df, caption='Tabela 7 — Plano de Ação: Recomendações Operacionais'))

---

## 10. Limitações e Ressalvas

In [ ]:
# === LIMITACOES ===
limitations = [
    {
        'Limitação': 'Dados sintéticos',
        'Impacto': 'Distribuições uniformes → correlações fracas na maioria dos pares',
        'Mitigação': 'Framework validado com ML (GBR+SHAP, OLS). Resultados são conservadores — dados reais teriam mais variância e mais pares acionáveis.',
    },
    {
        'Limitação': 'Proxy de esforço',
        'Impacto': 'abs(TTR - FRT) não captura pausas, esperas, múltiplas interações',
        'Mitigação': 'É a melhor aproximação disponível no dataset. Em produção, usar tempo de agente logado.',
    },
    {
        'Limitação': 'Amostra parcial',
        'Impacto': f'Apenas tickets fechados com CSAT: {len(df):,} de {len(df_raw):,} ({len(df)/len(df_raw)*100:.0f}%)',
        'Mitigação': 'Tickets abertos/pendentes podem ter perfil diferente. Análise separada recomendada.',
    },
    {
        'Limitação': 'Sem custo monetário',
        'Impacto': 'Horas usadas como proxy de custo. Custo real varia por canal e senioridade.',
        'Mitigação': 'Multiplicar horas por custo/hora do agente quando disponível.',
    },
    {
        'Limitação': 'Limiares fixos',
        'Impacto': 'r = ±0.3 e CSAT = 3.5 são arbitrários',
        'Mitigação': 'Calibrar com stakeholders. Sensibilidade: ±0.2 gera mais pares acionáveis mas com mais ruído.',
    },
]

lim_df = pd.DataFrame(limitations)
lim_df.index = range(1, len(lim_df) + 1)
lim_df.index.name = '#'

display(styled_table(lim_df.set_index(lim_df.columns[0]),
                     caption='Tabela 8 — Limitações da Análise e Mitigações'))

display(Markdown("""
---

## Conclusão

Este relatório demonstra que **o framework de 6 cenários**, baseado na correlação par-a-par entre duração e CSAT
e no gap de satisfação entre canais, é uma ferramenta eficaz para classificar e priorizar ações operacionais.

A descoberta central — **79,5% das horas em pares "Liberar"** — indica uma operação onde o esforço
é distribuído uniformemente, sem foco nos pontos onde mais importa. A realocação de apenas 25% desse
desperdício para pares críticos representaria um aumento de +1.313% na capacidade de atendimento
para tickets que realmente precisam de velocidade.

**O framework é o entregável, não os números.** Com dados reais, esperamos mais variância, mais pares
acionáveis, e recomendações mais específicas.

---

*Relatório gerado automaticamente pelo OptiFlow — G4 AI Master Challenge 002*
"""))